In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.7G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

# --- Single-region switch -------------------------------------------------
# Retarget the pipeline by changing variables.TARGET_REGION ("nsw"/"qld"/...).
REGION = variables.TARGET_REGION

PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES   # 12
PER_DAY  = 24 * PER_HOUR                                     # 288

# The bid stack IS the regional supply curve: for every in-region DUID we have
# the MW offered in each of 10 bands (_bands, from BIDPEROFFER_D) together with
# the $/MWh price of each band (_prices, from BIDDAYOFFER_D). Combining them
# gives how much capacity is offered at or below any price. Bids for interval T
# are known at (or before) T, so every feature here uses only current/past bids
# -- leakage-free. Price thresholds ($/MWh) define the cumulative supply buckets.
PRICE_THRESHOLDS = [0, 50, 100, 300, 1000, 5000]
FUEL_KEYS = {"Coal": "coal", "Gas": "gas", "Hydro": "hydro", "Battery Storage": "battery"}

_mapping = pd.read_parquet("../1_Dataset/Processed_data/0_nem_duid_mapping.parquet")
DUID_REGION = dict(zip(_mapping["DUID"], _mapping["Region"]))
DUID_FUEL   = dict(zip(_mapping["DUID"], _mapping["Fuel Source - Primary"]))


def _parse_bands(series: pd.Series) -> np.ndarray:
    """Parse a column of comma-separated 10-band strings into a (T, 10) array."""
    return series.str.split(",", expand=True).to_numpy(dtype=np.float32)

In [3]:
import pyarrow.parquet as pq

# Bids are stored per DUID: {DUID}_maxavail, {DUID}_bands (MW) in 8_bid_availability
# and {DUID}_prices ($/MWh) in 8_bid_prices. Prices come from the daily
# BIDDAYOFFER_D (forward-filled to 5-min); bands from 5-min BIDPEROFFER_D.
BANDS_PATH  = "../1_Dataset/Processed_data/8_bid_availability.parquet"
PRICES_PATH = "../1_Dataset/Processed_data/8_bid_prices.parquet"

# These files are ~572 comma-separated-string columns each; loading them whole
# expands to tens of GB of Python strings and is what crashes the kernel. The
# loop below only ever touches one DUID's columns at a time, so we read column
# names from the schema now and pull each DUID's data on demand.
bands_cols  = pq.ParquetFile(BANDS_PATH).schema_arrow.names
prices_cols = pq.ParquetFile(PRICES_PATH).schema_arrow.names

# In-region DUIDs that have both an availability and a price offer.
REGION_DUIDS = sorted(
    {c[:-len("_bands")]  for c in bands_cols  if c.endswith("_bands")}
    & {c[:-len("_prices")] for c in prices_cols if c.endswith("_prices")}
)
REGION_DUIDS = [d for d in REGION_DUIDS if DUID_REGION.get(d) == REGION.upper()]

# Use the complete dispatch-price timeline as the 5-minute spine. Sparse bid
# months are reindexed and causally forward-filled per DUID below.
bands_index = pd.read_parquet(
    "../1_Dataset/Processed_data/1_dispatch_price.parquet", columns=[]
).index

df = pd.DataFrame(index=bands_index)
df_core_columns = df.columns
print(f"{REGION}: {len(REGION_DUIDS)} DUIDs")
df[:10]


nsw: 110 DUIDs


""
Date
2018-01-01 00:05:00
2018-01-01 00:10:00
2018-01-01 00:15:00
2018-01-01 00:20:00
2018-01-01 00:25:00
2018-01-01 00:30:00
2018-01-01 00:35:00
2018-01-01 00:40:00
2018-01-01 00:45:00


In [4]:
def _add_supply_stack_features(index: pd.Index) -> pd.DataFrame:
    """
    Build the regional supply curve from the current bid stack. For every price
    threshold theta, sum the MW offered at or below theta across all in-region
    DUIDs (bands masked by their band prices). This traces how much cheap vs
    expensive capacity is on offer -- a direct, leakage-free read on how tight
    and how steep the supply stack is right now. Also totals offered MW, max
    available capacity, withheld capacity and a per-fuel breakdown (coal / gas /
    hydro / battery). Reads one DUID's columns at a time so peak memory stays at
    ~one DUID instead of the tens-of-GB full string frames.
    Returns only the new columns to avoid copying the full base frame.
    """
    n_rows = len(index)
    supply = {th: np.zeros(n_rows) for th in PRICE_THRESHOLDS}
    offered = np.zeros(n_rows)
    maxavail = np.zeros(n_rows)
    fuel_offered = {k: np.zeros(n_rows) for k in set(FUEL_KEYS.values())}
    fuel_cheap = {k: np.zeros(n_rows) for k in set(FUEL_KEYS.values())}

    for duid in REGION_DUIDS:
        max_col = f"{duid}_maxavail"
        b_cols  = [f"{duid}_bands"] + ([max_col] if max_col in bands_cols else [])
        bd = pd.read_parquet(BANDS_PATH, columns=b_cols).reindex(index).ffill()
        b  = _parse_bands(bd[f"{duid}_bands"])
        p_series = pd.read_parquet(PRICES_PATH, columns=[f"{duid}_prices"])[f"{duid}_prices"].reindex(index).ffill()
        p  = _parse_bands(p_series)
        m = min(len(b), len(p))
        b, p = b[:m], p[:m]

        band_total = np.nansum(b, axis=1)
        offered[:m] += band_total

        if max_col in bands_cols:
            maxavail[:m] += np.nan_to_num(bd[max_col].to_numpy(dtype=np.float64)[:m])

        for th in PRICE_THRESHOLDS:
            supply[th][:m] += np.nansum(np.where(p <= th, b, 0.0), axis=1)

        fuel_key = FUEL_KEYS.get(DUID_FUEL.get(duid))
        if fuel_key:
            fuel_offered[fuel_key][:m] += band_total
            fuel_cheap[fuel_key][:m] += np.nansum(np.where(p <= 300, b, 0.0), axis=1)

    new_cols = {}
    for th in PRICE_THRESHOLDS:
        new_cols[f"bidstack_{REGION}_mw_under_{th}"] = supply[th].astype(np.float32)
    new_cols[f"bidstack_{REGION}_offered_mw"]  = offered.astype(np.float32)
    new_cols[f"bidstack_{REGION}_maxavail_mw"] = maxavail.astype(np.float32)
    new_cols[f"bidstack_{REGION}_withheld_mw"] = (maxavail - offered).astype(np.float32)
    for k in fuel_offered:
        new_cols[f"bidstack_{REGION}_{k}_offered_mw"]    = fuel_offered[k].astype(np.float32)
        new_cols[f"bidstack_{REGION}_{k}_mw_under_300"] = fuel_cheap[k].astype(np.float32)

    return pd.DataFrame(new_cols, index=index)


new_df = _add_supply_stack_features(bands_index)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,bidstack_nsw_mw_under_0,bidstack_nsw_mw_under_50,bidstack_nsw_mw_under_100,bidstack_nsw_mw_under_300,bidstack_nsw_mw_under_1000,bidstack_nsw_mw_under_5000,bidstack_nsw_offered_mw,bidstack_nsw_maxavail_mw,bidstack_nsw_withheld_mw,bidstack_nsw_gas_offered_mw,bidstack_nsw_gas_mw_under_300,bidstack_nsw_hydro_offered_mw,bidstack_nsw_hydro_mw_under_300,bidstack_nsw_battery_offered_mw,bidstack_nsw_battery_mw_under_300,bidstack_nsw_coal_offered_mw,bidstack_nsw_coal_mw_under_300
Date,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11373.0,-4948.0,1998.0,440.0,4565.0,340.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:10:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11373.0,-4948.0,1998.0,440.0,4565.0,340.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:15:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11373.0,-4948.0,1998.0,440.0,4565.0,340.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:20:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11288.0,-5033.0,1998.0,440.0,4565.0,340.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:25:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11288.0,-5033.0,1998.0,440.0,4565.0,340.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:30:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11288.0,-5033.0,1998.0,440.0,4565.0,340.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:35:00,4488.0,4708.0,9523.0,9928.0,11288.0,11288.0,16121.0,11288.0,-4833.0,1998.0,440.0,4365.0,220.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:40:00,4488.0,4708.0,9523.0,9928.0,11288.0,11288.0,16121.0,11288.0,-4833.0,1998.0,440.0,4365.0,220.0,0.0,0.0,8560.0,8120.0
2018-01-01 00:45:00,4488.0,4708.0,9523.0,9928.0,11288.0,11288.0,16121.0,11288.0,-4833.0,1998.0,440.0,4365.0,220.0,0.0,0.0,8560.0,8120.0


In [5]:
def _add_bidstack_dynamics_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Economic-withholding ratios, high-price capacity share and rebidding
    dynamics derived from the supply-stack aggregates. Generators withholding
    capacity or re-offering MW from cheap into expensive price bands (versus the
    previous interval or the same time yesterday) is a classic leading indicator
    of a price event. All differences are backward-looking, hence leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    offered  = df[f"bidstack_{REGION}_offered_mw"]
    maxavail = df[f"bidstack_{REGION}_maxavail_mw"]
    under300 = df[f"bidstack_{REGION}_mw_under_300"]

    new_cols = {}
    new_cols[f"bidstack_{REGION}_withheld_ratio"]   = ((maxavail - offered) / (maxavail + 1)).clip(0, 1).astype(np.float32)
    new_cols[f"bidstack_{REGION}_high_priced_mw"]   = (offered - under300).clip(lower=0).astype(np.float32)
    new_cols[f"bidstack_{REGION}_high_priced_frac"] = ((offered - under300) / (offered + 1)).clip(0, 1).astype(np.float32)

    for lag, lab in [(1, "5m"), (PER_HOUR, "1h"), (PER_DAY, "1d")]:
        new_cols[f"bidstack_{REGION}_offered_rebid_{lab}"]  = offered.diff(lag).astype(np.float32)
        new_cols[f"bidstack_{REGION}_under300_rebid_{lab}"] = under300.diff(lag).astype(np.float32)
        new_cols[f"bidstack_{REGION}_withheld_rebid_{lab}"] = (maxavail - offered).diff(lag).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_bidstack_dynamics_features(df)
df = pd.concat([df, new_df], axis=1)
df[:10]


,bidstack_nsw_mw_under_0,bidstack_nsw_mw_under_50,bidstack_nsw_mw_under_100,bidstack_nsw_mw_under_300,bidstack_nsw_mw_under_1000,bidstack_nsw_mw_under_5000,bidstack_nsw_offered_mw,bidstack_nsw_maxavail_mw,bidstack_nsw_withheld_mw,bidstack_nsw_gas_offered_mw,...,bidstack_nsw_high_priced_frac,bidstack_nsw_offered_rebid_5m,bidstack_nsw_under300_rebid_5m,bidstack_nsw_withheld_rebid_5m,bidstack_nsw_offered_rebid_1h,bidstack_nsw_under300_rebid_1h,bidstack_nsw_withheld_rebid_1h,bidstack_nsw_offered_rebid_1d,bidstack_nsw_under300_rebid_1d,bidstack_nsw_withheld_rebid_1d
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11373.0,-4948.0,1998.0,...,0.384328,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11373.0,-4948.0,1998.0,...,0.384328,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11373.0,-4948.0,1998.0,...,0.384328,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11288.0,-5033.0,1998.0,...,0.384328,0.0,0.0,-85.0,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11288.0,-5033.0,1998.0,...,0.384328,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,4608.0,4828.0,9643.0,10048.0,11608.0,11608.0,16321.0,11288.0,-5033.0,1998.0,...,0.384328,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,4488.0,4708.0,9523.0,9928.0,11288.0,11288.0,16121.0,11288.0,-4833.0,1998.0,...,0.384133,-200.0,-120.0,200.0,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,4488.0,4708.0,9523.0,9928.0,11288.0,11288.0,16121.0,11288.0,-4833.0,1998.0,...,0.384133,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,4488.0,4708.0,9523.0,9928.0,11288.0,11288.0,16121.0,11288.0,-4833.0,1998.0,...,0.384133,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print("Total features:", df.shape[1])
df = df.drop(columns=df_core_columns)
df.to_parquet("../2_Features_build/Feature_data/8_1_bid_stack.parquet")
df.shape

Total features: 29


(893664, 29)

In [7]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 21 variable(s); kernel rss 0.58G, 9.3G RAM free now
